In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

## Ingestion del archivo "country.json"

###Paso 1 - Leer el archivo JSON usando "DataframeReader" de Spark

In [0]:
country_schema_df = "countryId INT, countryIsoCode STRING, countryName STRING"

In [0]:
country_df = spark.read.schema(country_schema_df).json(f"{bronze_folder_path}/{v_file_date}/country.json")

### Paso 2 - Eliminar las columnas no deseadas del DataFrame

In [0]:
countries_drpped_df = country_df.drop("countryIsoCode")
#countries_drpped_df = country_df.drop(country_df["countryIsoCode"])

### Paso 3 - Cambiar el nomre de las columnas y añadir "ingestion_date" y "environment"

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
countries_final_df = add_ingestion_date(countries_drpped_df) \
    .withColumnsRenamed({
        "countryId": "country_id","countryName": "country_name"
        }) \
    .withColumn("environment", lit(v_environment))\
    .withColumn("file_date", lit(v_file_date))

### Paso 4 - Escribir la salida en un formato "Parquet"

In [0]:
#countries_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/countries")

In [0]:
countries_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.countries")

In [0]:
%sql
SELECT file_date, count(1)
FROM movie_silver.countries 
GROUP BY file_date;

file_date,count(1)
2024-12-16,88


In [0]:
dbutils.notebook.exit("Exitoso")